# Correctness VerificationWe check that our pipeline recovers the expected Betti numbers on the four synthetic datasets, and compare each persistence diagram with the one produced by Ripser on the same point cloud.This notebook executes the pipeline directly: the datasets are small, so the computation is fast and the result is best seen visually.

## 1. Setup

In [ ]:
import syssys.path.insert(0, '../src')sys.path.insert(0, '../datasets')import matplotlib.pyplot as pltfrom ripser import ripserfrom persistent_homology import compute_persistencefrom persistent_homology.plotting import (    plot_persistence_diagram, ripser_to_diagram)from synthetic import make_circle, make_sphere, make_torus, make_cube

## 2. Helpers

In [ ]:
def betti_essential(diagram, dim):    """Number of essential (infinite) features in a dimension."""    return sum(1 for _, d in diagram.get(dim, [])               if d == float('inf'))def compare(points, max_dim, max_eps, annotations=None,            name='', figprefix=None):    """Compute our diagram and Ripser's, then plot them side by side."""    ours = compute_persistence(points, max_dim=max_dim + 1,                               max_eps=max_eps, algorithm='cohomology')    ours = {d: ivs for d, ivs in ours.items() if d <= max_dim}    res = ripser(points, maxdim=max_dim, thresh=max_eps)    theirs = ripser_to_diagram(res)    fig, axes = plt.subplots(1, 2, figsize=(13, 6))    plot_persistence_diagram(ours, max_dim=max_dim,                             annotations=annotations,                             title='Our implementation', ax=axes[0])    plot_persistence_diagram(theirs, max_dim=max_dim,                             title='Ripser', ax=axes[1])    fig.suptitle(name)    plt.tight_layout()    if figprefix:        fig.savefig(f'../figures/{figprefix}.png', dpi=150,                    bbox_inches='tight')    plt.show()    return ours, theirs

## 3. Circle $S^1$Expected: $\beta_0 = 1$, $\beta_1 = 1$.

In [ ]:
points = make_circle(n=30)ours, _ = compare(points, max_dim=1, max_eps=1.0,                  annotations=[{'dim': 1, 'xy': None,                                'text': 'the loop\n(essential)'}]                  if False else None,                  name='Noisy circle', figprefix='circle')print('beta_1 (essential):', betti_essential(ours, 1))

## 4. Sphere $S^2$Expected: $\beta_0 = 1$, $\beta_1 = 0$, $\beta_2 = 1$.

In [ ]:
points = make_sphere(n=100)ours, _ = compare(points, max_dim=2, max_eps=2.0,                  name='Sphere', figprefix='sphere')print('beta_2 (essential):', betti_essential(ours, 2))

## 5. Torus $T^2$Expected: $\beta_0 = 1$, $\beta_1 = 2$, $\beta_2 = 1$.

In [ ]:
points = make_torus(n_u=20, n_v=5)ours, _ = compare(points, max_dim=2, max_eps=2.0,                  name='Torus', figprefix='torus')print('beta_1 (essential):', betti_essential(ours, 1))

## 6. Random cube $[0,1]^3$Expected: $\beta_0 = 1$, $\beta_1 = 0$, $\beta_2 = 0$ (a contractible space).

In [ ]:
points = make_cube(n=100)ours, _ = compare(points, max_dim=2, max_eps=2.0,                  name='Random cube', figprefix='cube')print('beta_1, beta_2 (essential):',      betti_essential(ours, 1), betti_essential(ours, 2))